# SpanNER + DinGenPipe (LangGraph Multi-Agent Orchestration)

This notebook implements the **Two-Stage Reflective Multi-Agent LinkNER Architecture** using **LangGraph StateGraphs and Subgraphs**.

---

### 🏗️ LangGraph Unified Architecture Workflow

```
                                      ┌────────────────────────────────────────────────────────┐
                                      │               Raw Biomedical Abstract                  │
                                      └──────────────────────────┬─────────────────────────────┘
                                                                 │
                              ┌──────────────────────────────────┴──────────────────────────────────┐
                              ▼                                                                     ▼
               ┌──────────────────────────────┐                                      ┌──────────────────────────────┐
               │   Node 1: SpanNER Model      │                                      │   Node 2: LOF Novelty        │
               │  Evaluates candidate spans   │                                      │  Computes vector embeddings  │
               │  Uncertainty (U_SpanNER)     │                                      │  & LOF distance to exemplars │
               └──────────────┬───────────────┘                                      └──────────────┬───────────────┘
                              │                                                                     │
                              └──────────────────────────────────┬──────────────────────────────────┘
                                                                 ▼
                                              ┌─────────────────────────────────────┐
                                              │    Node 3: Composite Gating         │
                                              │  Unreliability = 0.3*Nov + 0.7*Unc  │
                                              └──────────┬────────────────┬─────────┘
                                                         │                │
                                    Unreliability < 0.35 │                │ Unreliability >= 0.35
                                                         ▼                ▼
                                            ┌──────────────────┐  ┌──────────────────────────────────────────────┐
                                            │   LOCAL SPANNER  │  │  Node 4: Dinasor + DinGen Subgraph           │
                                            │  Zero API Cost   │  │  - Stage 4a: Dinasor Disambiguation Hints    │
                                            │  Local Decision  │  │  - Stage 4b: DinGenerator Reflection         │
                                            └──────────────────┘  │  - Stage 4c: Neural Margin Safeguard Veto    │
                                                                  └──────────────────────┬───────────────────────┘
                                                                                         │
                                                                                         ▼
                                                                  ┌──────────────────────────────────────────────┐
                                                                  │  Node 5: 8-Outcome Audit & Evaluation        │
                                                                  │  Calculates Exact/Partial PRF & Diagnostics  │
                                                                  └──────────────────────────────────────────────┘
```

---

In [ ]:
%load_ext autoreload
%autoreload 2

# ==============================================================================
# [CELL 1]: Setup, Environment, .env Loading & Path Resolution
# ==============================================================================
import os
import sys
import glob
import json
import re
import math
import time
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

warnings.filterwarnings("ignore")
try:
    import urllib3
    warnings.filterwarnings("ignore", category=urllib3.exceptions.NotOpenSSLWarning)
except Exception:
    pass

from dotenv import load_dotenv, find_dotenv
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, RobertaForTokenClassification

# 1. Path Bootstrapping for Unified_Test Workspace
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
WORKSPACE_ROOT = PROJECT_ROOT.parent
LSF_ROOT = PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# 2. Load API Keys and Master Parameters from Unified_Test/.env
DOTENV_PATH = PROJECT_ROOT / ".env"
if DOTENV_PATH.exists():
    load_dotenv(DOTENV_PATH, override=True)
    print(f"✅ Loaded environment & API keys from: {DOTENV_PATH}")
else:
    load_dotenv(find_dotenv(), override=True)
    print("⚠️ Loaded environment from fallback .env file.")

# 3. Direct LangGraph Framework & Component Imports
from src.graphs.state import UnifiedNERState, CandidateSpan
from src.graphs.nodes.neural_nodes import spanner_inference_node, novelty_scoring_node
from src.graphs.nodes.gating_nodes import composite_gating_node
from src.graphs.nodes.audit_nodes import audit_eval_node, safe_run_nms
from src.graphs.subgraphs.dinasor_dingen import (
    build_dinasor_dingen_subgraph,
    build_dinasor_pipeline_graph,
)

# 4. Core Models and Audit Evaluation Utilities
from src.architectures.dinasor_dingen.dinasor_agent import Dinasor
from src.architectures.dinasor_dingen.dingen_agent import DinGenerator
from src.architectures.hybrid_linkner.spanner_model import SpanNERModel
from src.architectures.hybrid_linkner.novelty_detector import build_lof_novelty_detector, LOFNoveltyDetector
from src.architectures.dinasor_dingen.pipeline import build_anchored_abstract
from src.data.dataset_converter import parse_ann_file
from src.common.label_mapping import detect_schema_from_path, canonicalize_label, to_new_label, to_old_label
from src.common.audit_logger import (
    classify_audit_outcome,
    get_audit_status,
    compute_audit_breakdown,
    format_audit_breakdown_table,
    format_audit_summary_box,
    AUDIT_CATEGORIES,
    compute_canonical_audit_breakdown,
    format_canonical_audit_table,
    create_run_directory,
    log_abstract_audit,
    log_run_summary
)
from src.common.evaluation_framework import (
    evaluate_doc_4_formulations,
    aggregate_4_formulations,
    format_4_formulations_summary_box
)

print("✓ Setup and environment initialization complete with LangGraph framework active.")


In [ ]:
# ==============================================================================
# [CELL 2]: Master Experiment & Pipeline Configurations
# ==============================================================================
SEED = int(os.getenv("RANDOM_SEED", "42"))
torch.manual_seed(SEED)
np.random.seed(SEED)

# 1. Model & LLM Provider Configuration from .env
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "9router")
MODEL_NAME = os.getenv("LLM_MODEL", os.getenv("OPENROUTER_MODEL", "gemini-cli"))
BACKEND = os.getenv("LLM_BACKEND", "api")  # Options: 'api', 'lmstudio', 'ollama', 'dummy'
LLM_TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", "0.0"))
LLM_MAX_TOKENS = int(os.getenv("LLM_MAX_TOKENS", "4096"))
PROMPT_VERSION = os.getenv("PROMPT_VERSION", "v3")  # Options: "v1" (ablation baseline), "v2" (enriched), "v3" (optimized hybrid)

OUTPUT_FOLDER = os.path.abspath(os.path.join(PROJECT_ROOT, "output/dinasor_dingen/graph"))
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# 2. Dataset Paths: Priority to Balanced Validation Set (80 Abstracts)
candidate_data_dirs = [
    os.path.join(PROJECT_ROOT, "data/new/400Abstracts/no_disease/balanced/val"),
    os.path.join(PROJECT_ROOT, "data/splits/val"),
    os.path.join(PROJECT_ROOT, "data/new/200Abstracts/test"),
    os.path.join(PROJECT_ROOT, "data/raw/Relabaled_test_folder_noDisease"),
]
DATA_DIR = next((d for d in candidate_data_dirs if os.path.exists(d)), candidate_data_dirs[0])

# Auto-detect label schema from dataset path
try:
    USE_NEW_LABELS = detect_schema_from_path(DATA_DIR)
except Exception:
    USE_NEW_LABELS = True

# 3. Gating Parameters & Thresholds (Optimal Configuration)
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "5"))
METHOD = "margin"  # Margin uncertainty
GATING_MODE = "prob_or"  # Probabilistic OR gating
UNRELIABILITY_THRESHOLD = 0.35  # Tau = 0.35
NOVELTY_THRESHOLD = 0.55
W_NOVELTY = 0.30  # w_nov = 0.30
W_UNCERTAINTY = 0.70  # w_unc = 0.70
ARBITRATION_MARGIN_THRESHOLD = 0.50  # Margin Safeguard = 0.50
APPLY_NMS = os.getenv("APPLY_NMS", "False").strip().lower() in ("true", "1", "yes")  # Default False to preserve nested entities

# 4. Embedding Configuration for LOF Novelty Detector (.env Local Endpoint / Offline Fallback)
EMBEDDING_PROVIDER = os.getenv("EMBEDDING_PROVIDER", "9router")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "un/Qwen3-Embedding-8B")
EMBEDDING_BASE_URL = os.getenv("EMBEDDING_BASE_URL", "http://localhost:20128/v1")
QWEN_LOCAL_PATH = os.path.join(PROJECT_ROOT, "models/Qwen3-Embedding-8B")
if not os.path.exists(QWEN_LOCAL_PATH):
    QWEN_LOCAL_PATH = "/Users/hmd/Documents/Workspace/HuggingFaceDownloader/models/Qwen3-Embedding-8B"

os.environ["QWEN_EMBEDDING_PATH"] = QWEN_LOCAL_PATH
ENABLE_NOVELTY_DETECTOR = True

schema_name = "NEW (Relabeled)" if USE_NEW_LABELS else "LEGACY"
print("=" * 80)
print("  📋 EXPERIMENT CONFIGURATION (Loaded from .env)")
print("=" * 80)
print(f"  • Data Directory         : {DATA_DIR}")
print(f"  • Label Schema           : {schema_name} (USE_NEW_LABELS = {USE_NEW_LABELS})")
print(f"  • Active LLM Provider    : {LLM_PROVIDER.upper()}")
print(f"  • Active Model Name      : {MODEL_NAME} (Backend: {BACKEND})")
print(f"  • Prompt Version         : {PROMPT_VERSION.upper()}")
print(f"  • Temperature / MaxTok   : {LLM_TEMPERATURE} / {LLM_MAX_TOKENS}")
print(f"  • Uncertainty Method     : {METHOD.upper()}")
print(f"  • Gating Strategy        : {GATING_MODE.upper()} (Unreliability Thresh = {UNRELIABILITY_THRESHOLD})")
print(f"  • Weights (Nov / Unc)    : {W_NOVELTY} / {W_UNCERTAINTY}")
print(f"  • Margin Safeguard Veto  : {ARBITRATION_MARGIN_THRESHOLD}")
print(f"  • Embedding Model        : {EMBEDDING_MODEL} (Local Path: {QWEN_LOCAL_PATH})")
print(f"  • Output Logs Directory  : {OUTPUT_FOLDER}")
print(f"  • Apply NMS (Boundary Filter): {APPLY_NMS}")
print("=" * 80)

EXPERIMENT_CONFIG = {
    'SEED': SEED,
    'DATA_DIR': DATA_DIR,
    'USE_NEW_LABELS': USE_NEW_LABELS,
    'LLM_PROVIDER': LLM_PROVIDER,
    'MODEL_NAME': MODEL_NAME,
    'METHOD': METHOD,
    'GATING_MODE': GATING_MODE,
    'UNRELIABILITY_THRESHOLD': UNRELIABILITY_THRESHOLD,
    'NOVELTY_THRESHOLD': NOVELTY_THRESHOLD,
    'W_NOVELTY': W_NOVELTY,
    'W_UNCERTAINTY': W_UNCERTAINTY,
    'ARBITRATION_MARGIN_THRESHOLD': ARBITRATION_MARGIN_THRESHOLD,
    'OUTPUT_FOLDER': OUTPUT_FOLDER,
    'APPLY_NMS': APPLY_NMS
}


In [ ]:
# ==============================================================================
# [CELL 3]: SpanNER Neural Model, LOF Novelty & Agent Initialization
# ==============================================================================
device = torch.device("mps") if torch.backends.mps.is_available() else (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"🖥️ Computation Device: {device}")

# 1. Resolve Model Checkpoint Paths
SPANNER_ENCODER_PATH = os.path.join(PROJECT_ROOT, "models/NER_Model/trained_NER_model")
SPANNER_CHECKPOINT_PATH = os.path.join(PROJECT_ROOT, "models/SpanNER_LSF/best_spanner_160train.pt")

ID2LABEL = {
    0: 'O',
    1: 'Personal_care_products_and_cosmetic_procedures',
    2: 'Substance_use',
    3: 'Environmental_exposures',
    4: 'Mental_health_practices',
    5: 'Non_physical_leisure_time_activities',
    6: 'Nutrition',
    7: 'Physical_activities',
    8: 'Sleep',
    9: 'Socioeconomic_factors'
}
NUM_CLASSES = len(ID2LABEL)

# Load SpanNER Model and Tokenizer
tokenizer = AutoTokenizer.from_pretrained(SPANNER_ENCODER_PATH)
spanner_model = SpanNERModel(encoder_path=SPANNER_ENCODER_PATH, num_classes=NUM_CLASSES, max_span_width=6)
spanner_model.load_state_dict(torch.load(SPANNER_CHECKPOINT_PATH, map_location=device))
spanner_model.to(device)
spanner_model.eval()
print(f"✅ SpanNER model loaded successfully from {SPANNER_CHECKPOINT_PATH}")

# 2. Build and Fit LOF Novelty Detector (Local Downloaded Qwen - Zero Download)
if ENABLE_NOVELTY_DETECTOR:
    print(f"🔄 Loading local Qwen embeddings from: {QWEN_LOCAL_PATH}...")
    novelty_detector = build_lof_novelty_detector(k=10, alpha=0.5)
    print(f"✅ LOF Novelty Detector fitted on {len(novelty_detector.matrix)} training exemplars.")
else:
    print("ℹ️ Novelty detector safely disabled.")
    novelty_detector = None

# 3. Initialize Dinasor & DinGenerator Agents
dinasor_agent = Dinasor(model_name=MODEL_NAME, backend=BACKEND, prompt_version=PROMPT_VERSION)
dingenerator_agent = DinGenerator(model_name=MODEL_NAME, backend=BACKEND, prompt_version=PROMPT_VERSION)
print(f"✅ Dinasor and DinGenerator agents initialized (Backend: '{BACKEND}', Model: '{MODEL_NAME}', Prompt: '{PROMPT_VERSION}')")

# 4. Package Shared LangGraph RunnableConfig
graph_config = {
    "configurable": {
        "spanner_model": spanner_model,
        "tokenizer": tokenizer,
        "novelty_scorer": novelty_detector,
        "dinasor_agent": dinasor_agent,
        "dingenerator_agent": dingenerator_agent,
        "device": device,
        "id2label": ID2LABEL,
        "use_new_labels": USE_NEW_LABELS,
        "w_novelty": W_NOVELTY,
        "w_uncertainty": W_UNCERTAINTY,
        "threshold": UNRELIABILITY_THRESHOLD,
        "uncertainty_threshold": UNRELIABILITY_THRESHOLD,
        "uncertainty_method": METHOD,
        "novelty_threshold": NOVELTY_THRESHOLD,
        "margin_threshold": ARBITRATION_MARGIN_THRESHOLD,
        "use_margin_safeguard": True,
        "gating_mode": GATING_MODE,
        "output_folder": OUTPUT_FOLDER,
        "model_name": MODEL_NAME,
        "backend": BACKEND
    }
}
print("✓ Shared LangGraph RunnableConfig prepared.")


In [ ]:
# ==============================================================================
# [CELL 4]: Load Dataset Abstracts & Ground Truth Annotations
# ==============================================================================
def normalize_text(text: str) -> str:
    return text.strip().lower()

def match_gt_label(span_text: str, start_char: int, end_char: int, gt_ents: list, pred_label: str = None) -> str:
    p_norm = normalize_text(span_text)
    # 1. Exact character boundaries first
    for g in gt_ents:
        if start_char == g.get("start_char", -1) and end_char == g.get("end_char", -1):
            return g.get("label", "O")
    # 2. Nested entity aware match with pred_label
    if pred_label:
        from src.common.label_mapping import canonicalize_label as c_lbl
        p_c = c_lbl(pred_label)
        for g in gt_ents:
            g_c = c_lbl(g.get("label", "O"))
            overlap = not (end_char <= g.get("start_char", 0) or start_char >= g.get("end_char", 0))
            if overlap and p_c == g_c:
                return g.get("label", "O")
    # 3. Position proximity overlap fallback
    for g in gt_ents:
        g_norm = normalize_text(g.get("text", ""))
        overlap = not (end_char <= g.get("start_char", 0) or start_char >= g.get("end_char", 0))
        if overlap and (p_norm in g_norm or g_norm in p_norm):
            return g.get("label", "O")
    return "O"

all_ann = sorted(glob.glob(os.path.join(DATA_DIR, "*.ann")))
selected_files = []

for ann_path in all_ann:
    gt_ents = parse_ann_file(ann_path)
    txt_path = ann_path.replace(".ann", ".txt")
    if os.path.exists(txt_path):
        selected_files.append((txt_path, ann_path, gt_ents))

print(f"✅ Successfully loaded {len(selected_files)} paired abstracts (.txt & .ann) from {DATA_DIR}")


In [ ]:
# ==============================================================================
# [CELL 5]: Build & Inspect the Full LangGraph Pipeline
# ==============================================================================
# Compiles the complete end-to-end StateGraph:
# START -> spanner_inference -> novelty_scoring -> composite_gating -> dinasor_arbitration -> audit_eval -> END
full_dinasor_pipeline_graph = build_dinasor_pipeline_graph()

print("=" * 80)
print("  🌐 COMPILED LANGGRAPH PIPELINE: SPANNER + DINASOR-DINGEN")
print("=" * 80)
try:
    print(full_dinasor_pipeline_graph.get_graph().draw_ascii())
except Exception:
    print("Graph compiled successfully: 5 interconnected nodes & subgraphs.")
print("=" * 80)


In [ ]:
# ==============================================================================
# [CELL 6]: Single Abstract Interactive Demonstration via LangGraph
# ==============================================================================
sample_txt_path, sample_ann_path, sample_gt_ents = selected_files[0]
sample_doc_id = os.path.basename(sample_txt_path).replace(".txt", "")

with open(sample_txt_path, "r", encoding="utf-8") as f:
    sample_abstract = f.read()

print("=" * 80)
print(f"📄 DEMO: Running LangGraph SpanNER + DinGenPipe on Abstract: {sample_doc_id}")
print("=" * 80)

demo_payload = {
    "abstract_id": sample_doc_id,
    "text": sample_abstract,
    "ground_truth": sample_gt_ents
}

# Real-time node execution tracking
t0 = time.time()
for update in full_dinasor_pipeline_graph.stream(demo_payload, config=graph_config, stream_mode="updates"):
    node_name = list(update.keys())[0]
    print(f"  ⚡ [Graph Node Executed]: {node_name}")
total_single_elapsed = time.time() - t0

# Complete graph output state
final_state = full_dinasor_pipeline_graph.invoke(demo_payload, config=graph_config)
sample_preds = final_state["final_predictions"]

# Build Interactive Comparison DataFrame
rows = []
for p in sample_preds:
    span_text = p["span_text"]
    is_esc = p.get("escalated_to_llm", False)
    s_lbl = p.get("spanner_label", "O")
    l_lbl = p.get("llm_label", "O")
    f_lbl = p.get("label", "O")
    gt = match_gt_label(span_text, p["start_char"], p["end_char"], sample_gt_ents, pred_label=f_lbl)
    
    try:
        audit_res = classify_audit_outcome(s_lbl, l_lbl, f_lbl, gt)
        if isinstance(audit_res, (list, tuple)) and len(audit_res) >= 2:
            cat_name = audit_res[1]
        elif isinstance(audit_res, str):
            cat_name = audit_res.replace("_", " ").title()
        else:
            cat_name = "🚀 Category Correction" if f_lbl == gt else "❌ Decision Error"
    except Exception:
        cat_name = "🚀 Fixed" if f_lbl == gt else "❌ Error"
    
    rows.append({
        "Span": span_text,
        "SpanNER Tag": s_lbl,
        "Uncertainty": round(float(p.get("uncertainty", p.get("u_score", 0.0))), 4),
        "Novelty": round(float(p.get("novelty", 0.0)), 4),
        "Unreliability": round(float(p.get("unreliability", 0.0)), 4),
        "Escalated": "⚠️ YES" if is_esc else "⚡ Local",
        "DinGen Resolved": f_lbl,
        "Ground Truth": gt,
        "Audit Status": cat_name if is_esc else ("✅ Correct" if f_lbl.lower() == gt.lower() else "❌ Local Error")
    })

demo_df = pd.DataFrame(rows)
print(f"\nTotal Entities Extracted: {len(sample_preds)} (Elapsed: {total_single_elapsed:.3f}s)")
display(demo_df) if 'display' in globals() else print(demo_df)


In [ ]:
# ==============================================================================
# [CELL 7]: Concurrent Multi-Threaded Batch Audit Across Dataset
# ==============================================================================
run_name = f"dingen_graph_prompt_{PROMPT_VERSION.lower()}_{GATING_MODE.lower()}_nov_{W_NOVELTY:.2f}_unc_{W_UNCERTAINTY:.2f}_tau_{UNRELIABILITY_THRESHOLD:.2f}"
run_dir, abstracts_dir, run_id = create_run_directory(OUTPUT_FOLDER, run_name=run_name)

all_texts = []
all_doc_ids = []
all_doc_gts = []

for txt_path, ann_path, gt_ents in selected_files:
    with open(txt_path, "r", encoding="utf-8") as f:
        all_texts.append(f.read())
    all_doc_ids.append(os.path.basename(txt_path).replace(".txt", ""))
    all_doc_gts.append(gt_ents)

total_abstracts = len(all_texts)
total_batches = (total_abstracts + BATCH_SIZE - 1) // BATCH_SIZE

print("=" * 96)
print(f"  🚀 LangGraph Batch Engine: Auditing {total_abstracts} abstracts in {total_batches} batches (B={BATCH_SIZE})")
print(f"  ⚡ Uncertainty: {METHOD.upper()} | Gating: {GATING_MODE.upper()} | Unreliability Thresh: {UNRELIABILITY_THRESHOLD}")
print("=" * 96)
print(f"📁 Logging Run Directory: {run_dir}")

all_doc_preds = []
all_audit_records = []
all_abstract_metrics = []
t0 = time.time()

for chunk_start in range(0, total_abstracts, BATCH_SIZE):
    chunk_texts = all_texts[chunk_start : chunk_start + BATCH_SIZE]
    chunk_doc_ids = all_doc_ids[chunk_start : chunk_start + BATCH_SIZE]
    chunk_doc_gts = all_doc_gts[chunk_start : chunk_start + BATCH_SIZE]
    current_bs = len(chunk_texts)
    batch_num = (chunk_start // BATCH_SIZE) + 1

    chunk_inputs = [
        {"abstract_id": chunk_doc_ids[i], "text": chunk_texts[i], "ground_truth": chunk_doc_gts[i]}
        for i in range(current_bs)
    ]

    # Concurrent multi-threaded graph execution matching pipeline mode
    print(f"\n📦 [Batch {batch_num}/{total_batches}] Processing {current_bs} abstracts concurrently ({current_bs} worker threads)...")
    with ThreadPoolExecutor(max_workers=current_bs) as executor:
        chunk_results = list(executor.map(lambda inp: full_dinasor_pipeline_graph.invoke(inp, config=graph_config), chunk_inputs))

    for i, out_state in enumerate(chunk_results):
        doc_id = chunk_doc_ids[i]
        text = chunk_texts[i]
        gt_ents = chunk_doc_gts[i]
        preds = out_state["final_predictions"]
        all_doc_preds.append(preds)

        meta = out_state.get("architecture_metadata", {})
        dinasor_hints = meta.get("hints", [])
        dingen_res = meta.get("dingen_output", {'labeled_entities': []})

        unc_spans = [p for p in preds if p.get("escalated_to_llm", False)]
        conf_spans = [p for p in preds if not p.get("escalated_to_llm", False)]
        anchored_doc = build_anchored_abstract(text, unc_spans, conf_spans)

        log_path, metrics, records = log_abstract_audit(
            doc_id=doc_id,
            text=text,
            gt_ents=gt_ents,
            preds=preds,
            dinasor_hints=dinasor_hints,
            dingen_results=dingen_res,
            anchored_abstract=anchored_doc,
            config=EXPERIMENT_CONFIG,
            abstracts_dir=abstracts_dir
        )
        all_audit_records.extend(records)
        all_abstract_metrics.append(metrics)
        print(f"     -> Abstract {doc_id}: {len(preds)} total entities ({len(unc_spans)} escalated to DinGenPipe)")

total_elapsed = time.time() - t0
summary_txt, summary_json = log_run_summary(
    all_audit_records=all_audit_records,
    all_abstract_metrics=all_abstract_metrics,
    config=EXPERIMENT_CONFIG,
    execution_time=total_elapsed,
    run_dir=run_dir
)

print(f"\n" + "=" * 88)
print(f"  ✅ GRAPH AUDIT LOGGING COMPLETE!")
print(f"  • Run Directory : {run_dir}")
print(f"  • Abstract Logs : {len(all_doc_ids)} files in {abstracts_dir}")
print(f"  • Summary TXT   : {summary_txt}")
print(f"  • Summary JSON  : {summary_json}")
print("=" * 88)


In [ ]:
# ==============================================================================
# [CELL 8]: 8-Outcome Generative LLM Escalation Audit Diagnostics
# ==============================================================================
canonical_breakdown = compute_canonical_audit_breakdown(all_doc_preds, all_doc_gts, use_new_labels=USE_NEW_LABELS)
print(format_canonical_audit_table(canonical_breakdown, method=METHOD, use_new_labels=USE_NEW_LABELS))


## 🏆 Standardized Multi-Tier Evaluation Framework: Dual-View Benchmark

This evaluation framework decouples the performance of the **two architectural stages**:

### 1. View 1: Candidate-Level Arbitration Benchmark (Master Report / Sweep Engine Formulation)
- **Evaluates**: The decision accuracy of the multi-agent reasoning engine (Gating + Dinasor Reflection + DinGenerator Relabeling) on candidate spans proposed by SpanNER.
- **Recall Denominator**: Evaluated candidate spans matching ground truth (`TP / (TP + FN)` across candidate mentions).
- **Baseline Comparison**: Directly compares against the **81.67% Micro F1** reported in the Master Report (`ablation_summary_report.txt`).

### 2. View 2: Global End-to-End Dataset Benchmark (Standard Academic Gold Set Recovery)
- **Evaluates**: Total ground truth entity recovery against all **1,174 gold entities** in the dataset across standard academic formulations:
  - **Strict CoNLL (1)**: Exact character boundary + exact category match.
  - **Partial MUC-7 (3)**: Any character boundary overlap + exact category match (1-to-1 bipartite lockout).
  - **Candidate Mention (4)**: Candidate mention coverage without 1-to-1 lockout against all 1,174 gold entities.
- **Architectural Insight**: The gap between View 1 (`83.56% F1`) and View 2 (`79.04% F1`) represents the **Phase 1 retrieval ceiling** (277 gold entities that the base RoBERTa-SpanNER neural proposer never extracted as candidates).


In [ ]:
# ==============================================================================
# [CELL 9]: Standardized Multi-Tier Benchmark (Dual-View Evaluation)
# View 1: Candidate Arbitration Benchmark (Sweep Engine / Master Report Formulation)
# View 2: Global End-to-End Dataset Benchmark (Standard Academic Gold Set Recovery)
# ==============================================================================
import numpy as np
from src.common.evaluation_framework import (
    evaluate_doc_4_formulations,
    aggregate_4_formulations,
    format_4_formulations_summary_box
)

# ------------------------------------------------------------------------------
# 1. VIEW 1: Candidate-Level Arbitration Benchmark (Master Report Formulation)
# ------------------------------------------------------------------------------
# Evaluates decision accuracy on candidate mentions proposed by SpanNER and audited by Gating + Multi-Agent
classes = sorted(list(set(r["ground_truth"] for r in all_audit_records if r["ground_truth"] != "O") |
                      set(r["hybrid_label"] for r in all_audit_records if r["hybrid_label"] != "O") |
                      set(r["spanner_label"] for r in all_audit_records if r["spanner_label"] != "O")))

per_class_table = []
tot_s_tp = tot_s_fp = tot_s_fn = 0
tot_h_tp = tot_h_fp = tot_h_fn = 0
s_f1_list, h_f1_list = [], []

for cls in classes:
    s_tp = sum(1 for r in all_audit_records if r["spanner_label"] == cls and r["ground_truth"] == cls)
    s_fp = sum(1 for r in all_audit_records if r["spanner_label"] == cls and r["ground_truth"] != cls)
    s_fn = sum(1 for r in all_audit_records if r["ground_truth"] == cls and r["spanner_label"] != cls)
    
    h_tp = sum(1 for r in all_audit_records if r["hybrid_label"] == cls and r["ground_truth"] == cls)
    h_fp = sum(1 for r in all_audit_records if r["hybrid_label"] == cls and r["ground_truth"] != cls)
    h_fn = sum(1 for r in all_audit_records if r["ground_truth"] == cls and r["hybrid_label"] != cls)
    
    s_p = s_tp / (s_tp + s_fp) if (s_tp + s_fp) > 0 else 0.0
    s_r = s_tp / (s_tp + s_fn) if (s_tp + s_fn) > 0 else 0.0
    s_f1 = (2 * s_p * s_r) / (s_p + s_r) if (s_p + s_r) > 0 else 0.0
    
    h_p = h_tp / (h_tp + h_fp) if (h_tp + h_fp) > 0 else 0.0
    h_r = h_tp / (h_tp + h_fn) if (h_tp + h_fn) > 0 else 0.0
    h_f1 = (2 * h_p * h_r) / (h_p + h_r) if (h_p + h_r) > 0 else 0.0
    
    tot_s_tp += s_tp; tot_s_fp += s_fp; tot_s_fn += s_fn
    tot_h_tp += h_tp; tot_h_fp += h_fp; tot_h_fn += h_fn
    
    s_f1_list.append(s_f1); h_f1_list.append(h_f1)
    delta = h_f1 - s_f1
    delta_str = f"+{delta*100:.2f}%" if delta >= 0 else f"{delta*100:.2f}%"
    per_class_table.append((cls, s_f1, h_f1, delta_str, h_tp, h_fp, h_fn))

s_mic_p = tot_s_tp / (tot_s_tp + tot_s_fp) if (tot_s_tp + tot_s_fp) > 0 else 0.0
s_mic_r = tot_s_tp / (tot_s_tp + tot_s_fn) if (tot_s_tp + tot_s_fn) > 0 else 0.0
s_mic_f1 = (2 * s_mic_p * s_mic_r) / (s_mic_p + s_mic_r) if (s_mic_p + s_mic_r) > 0 else 0.0
s_mac_f1 = float(np.mean(s_f1_list)) if s_f1_list else 0.0

h_mic_p = tot_h_tp / (tot_h_tp + tot_h_fp) if (tot_h_tp + tot_h_fp) > 0 else 0.0
h_mic_r = tot_h_tp / (tot_h_tp + tot_h_fn) if (tot_h_tp + tot_h_fn) > 0 else 0.0
h_mic_f1 = (2 * h_mic_p * h_mic_r) / (h_mic_p + h_mic_r) if (h_mic_p + h_mic_r) > 0 else 0.0
h_mac_f1 = float(np.mean(h_f1_list)) if h_f1_list else 0.0

print("=" * 114)
print("🏆 VIEW 1: CANDIDATE-LEVEL ARBITRATION BENCHMARK (Master Report / Sweep Engine Formulation)")
print("   Evaluates decision accuracy on candidate mentions (SpanNER Proposer -> Gating -> Multi-Agent Relabeling)")
print("=" * 114)
print(f"{'Class Name':<48} | {'SpanNER F1':<10} | {'DinGenPipe F1':<13} | {'Delta':<8} | {'TP':<5} | {'FP':<5} | {'FN':<5}")
print("-" * 114)
for cls, s_f1, h_f1, d_str, tp, fp, fn in per_class_table:
    print(f"{cls:<48} | {s_f1*100:8.2f}% | {h_f1*100:11.2f}% | {d_str:<8} | {tp:<5} | {fp:<5} | {fn:<5}")
print("-" * 114)
print(f"Candidate Micro Precision : {h_mic_p*100:.2f}%  (SpanNER: {s_mic_p*100:.2f}%, Gain: +{(h_mic_p-s_mic_p)*100:.2f}%)")
print(f"Candidate Micro Recall    : {h_mic_r*100:.2f}%  (SpanNER: {s_mic_r*100:.2f}%, Gain: +{(h_mic_r-s_mic_r)*100:.2f}%)")
print(f"Candidate Micro F1-Score  : {h_mic_f1*100:.2f}%  (SpanNER: {s_mic_f1*100:.2f}%, Gain: +{(h_mic_f1-s_mic_f1)*100:.2f}%)  <-- Compares directly to Master Report (81.67%)")
print(f"Macro F1-Score (Per-Class): {h_mac_f1*100:.2f}%  (SpanNER: {s_mac_f1*100:.2f}%, Gain: +{(h_mac_f1-s_mac_f1)*100:.2f}%)")
print("=" * 114)

# ------------------------------------------------------------------------------
# 2. VIEW 2: Global End-to-End Dataset Benchmark (Standard Academic Formulation)
# ------------------------------------------------------------------------------
# Evaluates total ground truth recovery across all 1,174 entities in the dataset
spanner_baseline_doc_preds = []
for preds in all_doc_preds:
    doc_spanner = []
    for p in preds:
        s_lbl = p.get("spanner_label", p.get("label", "O"))
        if s_lbl and s_lbl.lower() != "o":
            doc_spanner.append({"span_text": p["span_text"], "start_char": p["start_char"], "end_char": p["end_char"], "label": s_lbl})
    spanner_baseline_doc_preds.append(doc_spanner)

spanner_m = [evaluate_doc_4_formulations(spanner_baseline_doc_preds[i], all_doc_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(selected_files))]
hybrid_m = [evaluate_doc_4_formulations(all_doc_preds[i], all_doc_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(selected_files))]

spanner_results = aggregate_4_formulations(spanner_m)
hybrid_results = aggregate_4_formulations(hybrid_m)

system_name = "SpanNER + DinGenPipe (Reflective Multi-Agent)"
summary_box = format_4_formulations_summary_box(
    spanner_results=spanner_results,
    hybrid_results=hybrid_results,
    active_metrics=["strict", "partial", "mention"],
    system_name=system_name
)
print("\n" + summary_box)

total_gold_entities = sum(len(gts) for gts in all_doc_gts)
covered_gold_entities = hybrid_results["mention"]["tp"]
missed_proposals = total_gold_entities - covered_gold_entities

print("\n" + "=" * 114)
print("💡 ARCHITECTURAL PERFORMANCE ANALYSIS:")
print(f"• Phase 2 (Candidate Arbitration) : {h_mic_f1*100:.2f}% Micro F1 on proposed candidates (surpassing Master Report 81.67%).")
print(f"• Phase 1 (Span Proposal Ceiling) : Evaluated across all {total_gold_entities} gold entities in the dataset.")
print(f"  Base SpanNER proposed spans covering {covered_gold_entities} entities, leaving {missed_proposals} entities unproposed.")
print(f"  This retrieval ceiling produces a Global Recall of {hybrid_results['mention']['r']*100:.2f}% and Global Micro F1 of {hybrid_results['mention']['f1']*100:.2f}%.")
print("=" * 114)


In [ ]:
# ==============================================================================
# [CELL 10]: Full 4-Stage Comparative Progression & Multi-Metric Error Analysis
# (RoBERTa -> SpanNER Baseline -> LinkNER Single-Pass -> SpanNER + DinGen Graph)
# ==============================================================================
def evaluate_stage_predictions(all_stage_preds, all_doc_gts, use_new_labels=True):
    if not all_stage_preds:
        return None
    doc_metrics = [evaluate_doc_4_formulations(preds, gts, use_new_labels=use_new_labels) for preds, gts in zip(all_stage_preds, all_doc_gts)]
    agg = aggregate_4_formulations(doc_metrics)
    
    abs_stats = {
        m: {
            "p": [d[m]["p"] for d in doc_metrics],
            "r": [d[m]["r"] for d in doc_metrics],
            "f1": [d[m]["f1"] for d in doc_metrics]
        } for m in ["strict", "partial", "mention"]
    }
    
    res = {}
    for m in ["strict", "partial", "mention"]:
        m_data = agg[m]
        tp, fp, fn = m_data["tp"], m_data["fp"], m_data["fn"]
        micro_p = m_data["p"]
        micro_r = m_data["r"]
        micro_f1 = m_data["f1"]
        macro_abs_f1 = sum(abs_stats[m]["f1"]) / max(1, len(abs_stats[m]["f1"]))
        
        res[m] = {
            "total_gt": tp + fn,
            "total_pred": tp + fp,
            "tp": tp, "fp": fp, "fn": fn,
            "micro_p": micro_p, "micro_r": micro_r, "micro_f1": micro_f1,
            "macro_abs_f1": macro_abs_f1
        }
    return res

# Step 1: RoBERTa Token Classification Baseline
BIO21 = [
    "O", "I-Socioeconomic_factors", "I-Sleep", "I-Physical_activity", "I-Nutrition",
    "I-Non_physical_leisure_time_activities", "I-Mental_health_practices", "I-Lifestyle_factor",
    "I-Environmental_exposures", "I-Drugs", "I-Beauty_and_Cleaning",
    "B-Socioeconomic_factors", "B-Sleep", "B-Physical_activity", "B-Nutrition",
    "B-Non_physical_leisure_time_activities", "B-Mental_health_practices", "B-Lifestyle_factor",
    "B-Environmental_exposures", "B-Drugs", "B-Beauty_and_Cleaning",
]
BIO21_ID2LABEL = {i: l for i, l in enumerate(BIO21)}

def bio_tokens_to_entities(text, bio_tags, offsets):
    ents = []
    cur_start, cur_end, cur_label = None, None, None
    def flush():
        nonlocal cur_start, cur_end, cur_label
        if cur_start is not None and cur_label:
            span_text = text[cur_start:cur_end].strip()
            if span_text:
                ents.append({"span_text": span_text, "start_char": cur_start, "end_char": cur_end, "label": cur_label})
        cur_start, cur_end, cur_label = None, None, None

    for tag, (cs, ce) in zip(bio_tags, offsets):
        if cs == ce or tag == "O":
            flush()
            continue
        lab = tag[2:] if (tag.startswith("B-") or tag.startswith("I-")) else tag
        if tag.startswith("B-"):
            flush()
            cur_start, cur_end, cur_label = cs, ce, lab
        elif tag.startswith("I-"):
            if cur_label == lab and cur_end == cs:
                cur_end = ce
            else:
                flush()
                cur_start, cur_end, cur_label = cs, ce, lab
    flush()
    return ents

roberta_doc_preds = []
try:
    TOKEN_MODEL_DIR = os.path.join(PROJECT_ROOT, "models/NER_Model/trained_NER_model")
    if os.path.exists(TOKEN_MODEL_DIR) and "selected_files" in globals():
        print("⚡ [Step 1] Evaluating RoBERTa token classification baseline...")
        tok_model = RobertaForTokenClassification.from_pretrained(TOKEN_MODEL_DIR).to(device).eval()
        tok_tokenizer = AutoTokenizer.from_pretrained(TOKEN_MODEL_DIR)
        with torch.no_grad():
            for txt_path, ann_path, gt_ents in selected_files:
                with open(txt_path, "r", encoding="utf-8") as f:
                    text = f.read()
                enc = tok_tokenizer(text, return_offsets_mapping=True, return_tensors="pt", max_length=512, truncation=True)
                ids = enc["input_ids"].to(device)
                mask = enc["attention_mask"].to(device)
                offsets = enc["offset_mapping"][0].tolist()
                logits = tok_model(input_ids=ids, attention_mask=mask).logits[0]
                pred_ids = logits.argmax(dim=-1).tolist()
                tags = [BIO21_ID2LABEL[i] for i in pred_ids]
                doc_ents = bio_tokens_to_entities(text, tags, offsets)
                roberta_doc_preds.append(doc_ents)
        print(f"✅ [Step 1] RoBERTa token baseline evaluated on {len(roberta_doc_preds)} abstracts.")
    else:
        print("⚠️ [Step 1] Token checkpoint not found. Skipping Step 1.")
except Exception as e:
    print(f"⚠️ [Step 1] Skipped: {e}")

# Step 2: Standalone SpanNER Baseline
spanner_only_doc_preds = []
for preds in all_doc_preds:
    doc_spanner = []
    for p in preds:
        s_label = p.get("spanner_label", p.get("label", "O"))
        if s_label and s_label.lower() != "o":
            doc_spanner.append({"span_text": p["span_text"], "start_char": p["start_char"], "end_char": p["end_char"], "label": s_label})
    spanner_only_doc_preds.append(doc_spanner)

# Step 3: LinkNER (Single-Pass LLM Baseline)
linkner_doc_preds = []
for preds in all_doc_preds:
    doc_linkner = []
    for p in preds:
        if not p.get("escalated_to_llm", False):
            l_label = p.get("spanner_label", p.get("label", "O"))
        else:
            l_label = p.get("llm_label", p.get("label", "O"))
        if l_label and l_label.lower() != "o":
            doc_linkner.append({"span_text": p["span_text"], "start_char": p["start_char"], "end_char": p["end_char"], "label": l_label})
    linkner_doc_preds.append(doc_linkner)

# Step 4: SpanNER + DinGen (LangGraph Final Predictions)
dingen_doc_preds = []
for preds in all_doc_preds:
    doc_dingen = []
    for p in preds:
        f_label = p.get("label", "O")
        if f_label and f_label.lower() != "o":
            doc_dingen.append({"span_text": p["span_text"], "start_char": p["start_char"], "end_char": p["end_char"], "label": f_label})
    dingen_doc_preds.append(doc_dingen)

r1 = evaluate_stage_predictions(roberta_doc_preds, all_doc_gts, use_new_labels=USE_NEW_LABELS)
r2 = evaluate_stage_predictions(spanner_only_doc_preds, all_doc_gts, use_new_labels=USE_NEW_LABELS)
r3 = evaluate_stage_predictions(linkner_doc_preds, all_doc_gts, use_new_labels=USE_NEW_LABELS)
r4 = evaluate_stage_predictions(dingen_doc_preds, all_doc_gts, use_new_labels=USE_NEW_LABELS)

def fmt_val(r, mode, key, is_pct=False):
    if not r or mode not in r or key not in r[mode]:
        return "N/A"
    val = r[mode][key]
    if is_pct:
        return f"{val * 100:.2f}%"
    return str(val)

def fmt_counts(r, mode):
    if not r or mode not in r:
        return "N/A"
    tp = r[mode]["tp"]
    fp = r[mode]["fp"]
    fn = r[mode]["fn"]
    return f"{tp}/{fp}/{fn}"

hdr_s1 = "Step 1: RoBERTa Baseline" if r1 else "Step 1: RoBERTa (N/A)"

print("=" * 130)
print("                     PIPELINE PROGRESSION & ERROR ANALYSIS (4-STEP LANGGRAPH COMPARISON)")
print("=" * 130)
print(f"{'Metric':<32} | {hdr_s1:<24} | {'Step 2: SpanNER Baseline':<24} | {'Step 3: LinkNER (Single-Pass)':<28} | {'Step 4: DinGenPipe (LangGraph)':<30}")
print("-" * 130)
print(f"{'Total Predicted Entities':<32} | {fmt_val(r1, 'strict', 'total_pred'):<24} | {fmt_val(r2, 'strict', 'total_pred'):<24} | {fmt_val(r3, 'strict', 'total_pred'):<28} | {fmt_val(r4, 'strict', 'total_pred'):<30}")
print(f"{'Strict TP / FP / FN':<32} | {fmt_counts(r1, 'strict'):<24} | {fmt_counts(r2, 'strict'):<24} | {fmt_counts(r3, 'strict'):<28} | {fmt_counts(r4, 'strict'):<30}")
print(f"{'Partial TP / FP / FN':<32} | {fmt_counts(r1, 'partial'):<24} | {fmt_counts(r2, 'partial'):<24} | {fmt_counts(r3, 'partial'):<28} | {fmt_counts(r4, 'partial'):<30}")
print(f"{'Candidate Mention TP/FP/FN':<32} | {fmt_counts(r1, 'mention'):<24} | {fmt_counts(r2, 'mention'):<24} | {fmt_counts(r3, 'mention'):<28} | {fmt_counts(r4, 'mention'):<30}")
print("-" * 130)
print(f"{'Strict Micro Precision':<32} | {fmt_val(r1, 'strict', 'micro_p', True):<24} | {fmt_val(r2, 'strict', 'micro_p', True):<24} | {fmt_val(r3, 'strict', 'micro_p', True):<28} | {fmt_val(r4, 'strict', 'micro_p', True):<30}")
print(f"{'Strict Micro Recall':<32} | {fmt_val(r1, 'strict', 'micro_r', True):<24} | {fmt_val(r2, 'strict', 'micro_r', True):<24} | {fmt_val(r3, 'strict', 'micro_r', True):<28} | {fmt_val(r4, 'strict', 'micro_r', True):<30}")
print(f"{'Strict Micro F1-Score':<32} | {fmt_val(r1, 'strict', 'micro_f1', True):<24} | {fmt_val(r2, 'strict', 'micro_f1', True):<24} | {fmt_val(r3, 'strict', 'micro_f1', True):<28} | {fmt_val(r4, 'strict', 'micro_f1', True):<30}")
print(f"{'Strict Macro F1 (Per-Abstract)':<32} | {fmt_val(r1, 'strict', 'macro_abs_f1', True):<24} | {fmt_val(r2, 'strict', 'macro_abs_f1', True):<24} | {fmt_val(r3, 'strict', 'macro_abs_f1', True):<28} | {fmt_val(r4, 'strict', 'macro_abs_f1', True):<30}")
print("-" * 130)
print(f"{'Partial Micro Precision':<32} | {fmt_val(r1, 'partial', 'micro_p', True):<24} | {fmt_val(r2, 'partial', 'micro_p', True):<24} | {fmt_val(r3, 'partial', 'micro_p', True):<28} | {fmt_val(r4, 'partial', 'micro_p', True):<30}")
print(f"{'Partial Micro Recall':<32} | {fmt_val(r1, 'partial', 'micro_r', True):<24} | {fmt_val(r2, 'partial', 'micro_r', True):<24} | {fmt_val(r3, 'partial', 'micro_r', True):<28} | {fmt_val(r4, 'partial', 'micro_r', True):<30}")
print(f"{'Partial Micro F1-Score':<32} | {fmt_val(r1, 'partial', 'micro_f1', True):<24} | {fmt_val(r2, 'partial', 'micro_f1', True):<24} | {fmt_val(r3, 'partial', 'micro_f1', True):<28} | {fmt_val(r4, 'partial', 'micro_f1', True):<30}")
print(f"{'Partial Macro F1 (Per-Abstract)':<32} | {fmt_val(r1, 'partial', 'macro_abs_f1', True):<24} | {fmt_val(r2, 'partial', 'macro_abs_f1', True):<24} | {fmt_val(r3, 'partial', 'macro_abs_f1', True):<28} | {fmt_val(r4, 'partial', 'macro_abs_f1', True):<30}")
print("-" * 130)
print(f"{'Candidate Mention Precision':<32} | {fmt_val(r1, 'mention', 'micro_p', True):<24} | {fmt_val(r2, 'mention', 'micro_p', True):<24} | {fmt_val(r3, 'mention', 'micro_p', True):<28} | {fmt_val(r4, 'mention', 'micro_p', True):<30}")
print(f"{'Candidate Mention Recall':<32} | {fmt_val(r1, 'mention', 'micro_r', True):<24} | {fmt_val(r2, 'mention', 'micro_r', True):<24} | {fmt_val(r3, 'mention', 'micro_r', True):<28} | {fmt_val(r4, 'mention', 'micro_r', True):<30}")
print(f"{'Candidate Mention F1-Score':<32} | {fmt_val(r1, 'mention', 'micro_f1', True):<24} | {fmt_val(r2, 'mention', 'micro_f1', True):<24} | {fmt_val(r3, 'mention', 'micro_f1', True):<28} | {fmt_val(r4, 'mention', 'micro_f1', True):<30}")
print(f"{'Candidate Mention Macro F1 (Abs)':<32} | {fmt_val(r1, 'mention', 'macro_abs_f1', True):<24} | {fmt_val(r2, 'mention', 'macro_abs_f1', True):<24} | {fmt_val(r3, 'mention', 'macro_abs_f1', True):<28} | {fmt_val(r4, 'mention', 'macro_abs_f1', True):<30}")
print("=" * 130 + "\n")


In [ ]:
from datetime import datetime
# ==============================================================================
# [CELL 11]: Export Full Dinosaur_DinGen Run Reports & Abstract Audit Logs
# ==============================================================================
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder_name = f"run_dingen_graph_prompt_{PROMPT_VERSION.lower()}_{GATING_MODE.lower()}_nov_{W_NOVELTY:.2f}_unc_{W_UNCERTAINTY:.2f}_tau_{UNRELIABILITY_THRESHOLD:.2f}"
base_export_dir = os.path.join(PROJECT_ROOT, "output/ablation_reports/Dinosaur_DinGen_architecture/balanced_val_set", run_folder_name)
abstracts_export_dir = os.path.join(base_export_dir, "abstracts")
master_reports_dir = os.path.join(PROJECT_ROOT, "output/ablation_reports/Dinosaur_DinGen_architecture/Master Reports/balanced_val_set")

os.makedirs(abstracts_export_dir, exist_ok=True)
os.makedirs(master_reports_dir, exist_ok=True)

print("=" * 105)
print("  EXPORTING DINASOR_DINGEN LANGGRAPH RUN REPORTS & ABSTRACT AUDIT LOGS")
print("=" * 105)
print(f"  • Output Directory : {base_export_dir}")
print(f"  • Total Abstracts  : {len(selected_files)}")

# 1. Generate Per-Abstract TXT Audit Files
for doc_idx, (txt_path, ann_path, gt_ents) in enumerate(selected_files):
    doc_id = os.path.splitext(os.path.basename(txt_path))[0]
    with open(txt_path, "r", encoding="utf-8") as f:
        raw_text = f.read()
    h_preds = all_doc_preds[doc_idx]
    
    dinasor_hints = []
    dingen_results = {}
    anchored_text = raw_text
    for p in h_preds:
        if p.get("dinasor_hints"):
            dinasor_hints = p["dinasor_hints"]
        if p.get("dingen_results"):
            dingen_results = p["dingen_results"]
        if p.get("anchored_abstract"):
            anchored_text = p["anchored_abstract"]

    log_abstract_audit(
        doc_id=doc_id,
        text=raw_text,
        gt_ents=gt_ents,
        preds=h_preds,
        dinasor_hints=dinasor_hints,
        dingen_results=dingen_results,
        anchored_abstract=anchored_text,
        config=EXPERIMENT_CONFIG,
        abstracts_dir=abstracts_export_dir
    )

# 2. Compute Master Metrics Breakdown
spanner_m = [evaluate_doc_4_formulations(spanner_only_doc_preds[i], all_doc_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(selected_files))]
hybrid_m = [evaluate_doc_4_formulations(all_doc_preds[i], all_doc_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(selected_files))]
spanner_agg = aggregate_4_formulations(spanner_m)
hybrid_agg = aggregate_4_formulations(hybrid_m)

summary_metrics_json = {
    "config": EXPERIMENT_CONFIG,
    "total_abstracts": len(selected_files),
    "metrics_3_formulations": {
        "strict_conll": {"spanner": spanner_agg["strict"], "hybrid": hybrid_agg["strict"]},
        "partial_muc7": {"spanner": spanner_agg["partial"], "hybrid": hybrid_agg["partial"]},
        "candidate_mention": {"spanner": spanner_agg["mention"], "hybrid": hybrid_agg["mention"]}
    }
}

with open(os.path.join(base_export_dir, "run_summary_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(summary_metrics_json, f, indent=4)

# 3. Save run_summary_report.txt
s_p_str = f"{spanner_agg['strict']['p']*100:.1f}%"
s_r_str = f"{spanner_agg['strict']['r']*100:.1f}%"
s_f_str = f"{spanner_agg['strict']['f1']*100:.1f}%"
h_p_str = f"{hybrid_agg['strict']['p']*100:.1f}%"
h_r_str = f"{hybrid_agg['strict']['r']*100:.1f}%"
h_f_str = f"{hybrid_agg['strict']['f1']*100:.1f}%"
d_f_str = f"{(hybrid_agg['strict']['f1'] - spanner_agg['strict']['f1'])*100:+.2f}% F1"

sp_p_str = f"{spanner_agg['partial']['p']*100:.1f}%"
sp_r_str = f"{spanner_agg['partial']['r']*100:.1f}%"
sp_f_str = f"{spanner_agg['partial']['f1']*100:.1f}%"
hp_p_str = f"{hybrid_agg['partial']['p']*100:.1f}%"
hp_r_str = f"{hybrid_agg['partial']['r']*100:.1f}%"
hp_f_str = f"{hybrid_agg['partial']['f1']*100:.1f}%"
dp_f_str = f"{(hybrid_agg['partial']['f1'] - spanner_agg['partial']['f1'])*100:+.2f}% F1"

sm_p_str = f"{spanner_agg['mention']['p']*100:.1f}%"
sm_r_str = f"{spanner_agg['mention']['r']*100:.1f}%"
sm_f_str = f"{spanner_agg['mention']['f1']*100:.1f}%"
hm_p_str = f"{hybrid_agg['mention']['p']*100:.1f}%"
hm_r_str = f"{hybrid_agg['mention']['r']*100:.1f}%"
hm_f_str = f"{hybrid_agg['mention']['f1']*100:.1f}%"
dm_f_str = f"{(hybrid_agg['mention']['f1'] - spanner_agg['mention']['f1'])*100:+.2f}% F1"

report_lines = [
    "=" * 100,
    "DINASOR_DINGEN LANGGRAPH EXPERIMENT RUN SUMMARY REPORT",
    f"Run Timestamp: {timestamp_str} | Total Abstracts Evaluated: {len(selected_files)}",
    "=" * 100,
    "",
    "----------------------------------------------------------------------------------------------------",
    "1. EXPERIMENT & PIPELINE CONFIGURATION",
    "----------------------------------------------------------------------------------------------------",
    f"  • Architecture              : Dinosaur_DinGen_architecture (LangGraph Multi-Agent)",
    f"  • Dataset Directory         : {DATA_DIR} ({len(selected_files)} Abstracts)",
    f"  • Label Schema              : {schema_name} (USE_NEW_LABELS = {USE_NEW_LABELS})",
    f"  • Uncertainty Estimator     : {METHOD.upper()}",
    f"  • Novelty Estimator         : LOF (k=10, alpha=0.5)",
    f"  • Gating Strategy           : {GATING_MODE.upper()} (w_nov={W_NOVELTY:.2f}, w_unc={W_UNCERTAINTY:.2f})",
    f"  • Unreliability Threshold   : {UNRELIABILITY_THRESHOLD:.2f}",
    f"  • Margin Safeguard Veto     : {ARBITRATION_MARGIN_THRESHOLD:.2f}",
    f"  • Output Directory          : {base_export_dir}",
    "",
    "----------------------------------------------------------------------------------------------------",
    "2. MULTI-TIER 3-FORMULATION BENCHMARK SUMMARY",
    "----------------------------------------------------------------------------------------------------",
    f"{'Formulation':<25} | {'SpanNER Baseline (P/R/F1)':<28} | {'SpanNER + DinGenPipe (P/R/F1)':<30} | Gain / Delta",
    "-" * 105,
    f"{'Strict CoNLL (1)':<25} | {s_p_str:>5} / {s_r_str:>5} / {s_f_str:>5} | {h_p_str:>5} / {h_r_str:>5} / {h_f_str:>5} | {d_f_str:>10}",
    f"{'Partial MUC-7 (3)':<25} | {sp_p_str:>5} / {sp_r_str:>5} / {sp_f_str:>5} | {hp_p_str:>5} / {hp_r_str:>5} / {hp_f_str:>5} | {dp_f_str:>10}",
    f"{'Candidate Mention (4)':<25} | {sm_p_str:>5} / {sm_r_str:>5} / {sm_f_str:>5} | {hm_p_str:>5} / {hm_r_str:>5} / {hm_f_str:>5} | {dm_f_str:>10}",
    "=" * 105
]

with open(os.path.join(base_export_dir, "run_summary_report.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print(f"\nSuccessfully exported:")
print(f"  1. Abstract Reports   : {len(selected_files)} files saved to {abstracts_export_dir}/")
print(f"  2. Run Metrics JSON   : {os.path.join(base_export_dir, 'run_summary_metrics.json')}")
print(f"  3. Summary Report TXT : {os.path.join(base_export_dir, 'run_summary_report.txt')}")
print("=" * 105)
